In [1]:
import onnxscript
import torch
import torch.nn as nn
from torchvision import models, transforms
from PIL import Image
import numpy as np
import onnx
import onnxruntime as ort
import time
import os
from onnx import shape_inference

device = "cpu"  # we'll deliberately use CPU to demonstrate quantisation gains
torch.manual_seed(42)
from inference import FlowerClassifier
import os

In [2]:
model = models.resnet18(weights=None)
model.fc = nn.Linear(model.fc.in_features, 102)
model.load_state_dict(torch.load("flowers102_resnet18.pth", map_location=device))
model.eval()

ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    )
    (1): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
  

## Task 1 — Export to ONNX and Verify

### Part A — Export

In [3]:
for f in ["flowers_resnet18.onnx", "flowers_resnet18.onnx.data", "flowers_resnet18.int8.onnx"]:
    if os.path.exists(f):
        os.remove(f)
        print(f"Silindi: {f}")

example = torch.randn(1, 3, 224, 224)
torch.onnx.export(
    model, 
    example, 
    "flowers_resnet18.onnx",
    input_names=["input"], 
    output_names=["logits"],
    dynamic_axes={"input": {0: "batch"}, "logits": {0: "batch"}},
    opset_version=17,
    dynamo=False, 
)
print("ONNX model is valid and weights are embedded.")
print(f"Size: {os.path.getsize('flowers_resnet18.onnx') / (1024 * 1024):.2f} MB")

Silindi: flowers_resnet18.onnx
Silindi: flowers_resnet18.int8.onnx


C:\Users\user\AppData\Local\Temp\ipykernel_9760\2519763025.py:7: DeprecationWarning: You are using the legacy TorchScript-based ONNX export. Starting in PyTorch 2.9, the new torch.export-based ONNX exporter has become the default. Learn more about the new export logic: https://docs.pytorch.org/docs/stable/onnx_export.html. For exporting control flow: https://pytorch.org/tutorials/beginner/onnx/export_control_flow_model_to_onnx_tutorial.html
  torch.onnx.export(


ONNX model is valid and weights are embedded.
Size: 42.83 MB


### Part B — Numerical equivalence check

In [4]:
session = ort.InferenceSession("flowers_resnet18.onnx")
random_images = torch.randn(8, 3, 224, 224)

model.eval()
with torch.no_grad():
    pytorch_out = model(random_images).numpy()

onnx_out = session.run(None, {"input": random_images.numpy()})[0]

max_diff = np.abs(pytorch_out - onnx_out).max()
print(f"Maximum absolute difference: {max_diff:.8f}")
assert max_diff < 1e-4, "Error! The difference is greater than 1e-4."
print("Success! Both models are numerically equivalent.")

Maximum absolute difference: 0.00000572
Success! Both models are numerically equivalent.


## Task 2 — Build an Inference Pipeline

In [5]:
classifier = FlowerClassifier("flowers_resnet18.onnx")
img_dir = "./data/flowers-102/jpg" # Sənin qovluq yolun
all_images = [f for f in os.listdir(img_dir) if f.endswith('.jpg')]
test_images = [os.path.join(img_dir, f) for f in all_images[:5]]

for img_path in test_images:
    print(f"\n--- Image: {os.path.basename(img_path)} ---")
    preds = classifier.predict(img_path, k=3)
    for i, (class_id, prob) in enumerate(preds):
        print(f"{i+1}. Class ID: {class_id} | Probability: {prob:.4f}")


--- Image: image_00001.jpg ---
1. Class ID: 76 | Probability: 0.9967
2. Class ID: 70 | Probability: 0.0033
3. Class ID: 9 | Probability: 0.0000

--- Image: image_00002.jpg ---
1. Class ID: 76 | Probability: 0.9943
2. Class ID: 70 | Probability: 0.0014
3. Class ID: 12 | Probability: 0.0008

--- Image: image_00003.jpg ---
1. Class ID: 76 | Probability: 0.9907
2. Class ID: 70 | Probability: 0.0058
3. Class ID: 12 | Probability: 0.0004

--- Image: image_00004.jpg ---
1. Class ID: 76 | Probability: 0.5157
2. Class ID: 37 | Probability: 0.2890
3. Class ID: 53 | Probability: 0.0298

--- Image: image_00005.jpg ---
1. Class ID: 76 | Probability: 0.9994
2. Class ID: 9 | Probability: 0.0001
3. Class ID: 61 | Probability: 0.0001


## Task 3 — Quantise to INT8 and Benchmark All Three Variants

In [7]:
from onnxruntime.quantization import quantize_dynamic, QuantType

quantize_dynamic(
    model_input="flowers_resnet18.onnx",
    model_output="flowers_resnet18.int8.onnx",
    weight_type=QuantType.QInt8,
)

In [8]:
session_int8 = ort.InferenceSession("flowers_resnet18.int8.onnx")
all_images = [f for f in os.listdir(img_dir) if f.endswith('.jpg')]
val_images = [os.path.join(img_dir, f) for f in all_images[:100]]

fp32_preds = []
int8_preds = []
true_labels = [] 

for img_path in val_images:
    x = classifier.preprocess(img_path)
    
    fp32_out = session.run(None, {"input": x})[0][0]
    int8_out = session_int8.run(None, {"input": x})[0][0]
    
    fp32_preds.append(fp32_out)
    int8_preds.append(int8_out)

fp32_preds = np.array(fp32_preds)
int8_preds = np.array(int8_preds)

max_diff = np.abs(fp32_preds - int8_preds).max()
mean_diff = np.abs(fp32_preds - int8_preds).mean()
print(f"Max absolute difference:  {max_diff:.6f}")
print(f"Mean absolute difference: {mean_diff:.6f}")

fp32_top1 = np.argmax(fp32_preds, axis=1)
int8_top1 = np.argmax(int8_preds, axis=1)
agreement = (fp32_top1 == int8_top1).mean() * 100
print(f"FP32 vs INT8 top-1 agreement: {agreement:.1f}%")

Max absolute difference:  1.670287
Mean absolute difference: 0.316379
FP32 vs INT8 top-1 agreement: 97.0%


INT8 quantization significantly reduces the model size. The maximum absolute difference between FP32 and INT8 model outputs is 1.67, with a mean difference of 0.32 — these are at the logit level, i.e. before softmax, which is why they appear large. However, a 97% top-1 prediction agreement shows that quantization has virtually no impact on actual classification decisions. This trade-off is acceptable for production use — in exchange for a 3% disagreement rate, the model size and inference speed improve significantly.

In [ ]:
session_fp32 = ort.InferenceSession("flowers_resnet18.onnx")
session_int8 = ort.InferenceSession("flowers_resnet18.int8.onnx")

test_input = torch.randn(1, 3, 224, 224)
test_np = test_input.numpy()

N = 100
model.eval()
with torch.no_grad():
    for _ in range(10): model(test_input)
t0 = time.perf_counter()
with torch.no_grad():
    for _ in range(N): model(test_input)
pytorch_ms = (time.perf_counter() - t0) / N * 1000

for _ in range(10): session_fp32.run(None, {"input": test_np})
t0 = time.perf_counter()
for _ in range(N): session_fp32.run(None, {"input": test_np})
fp32_ms = (time.perf_counter() - t0) / N * 1000

for _ in range(10): session_int8.run(None, {"input": test_np})
t0 = time.perf_counter()
for _ in range(N): session_int8.run(None, {"input": test_np})
int8_ms = (time.perf_counter() - t0) / N * 1000

fp32_size = os.path.getsize("flowers_resnet18.onnx") / (1024 * 1024)
int8_size = os.path.getsize("flowers_resnet18.int8.onnx") / (1024 * 1024)

print(f"{'Model':<20} {'Size (MB)':<15} {'Latency (ms)':<15} {'Speedup'}")
print("-" * 60)
print(f"{'PyTorch (FP32)':<20} {'—':<15} {pytorch_ms:<15.2f} 1.00x")
print(f"{'ONNX (FP32)':<20} {fp32_size:<15.2f} {fp32_ms:<15.2f} {pytorch_ms/fp32_ms:.2f}x")
print(f"{'ONNX (INT8)':<20} {int8_size:<15.2f} {int8_ms:<15.2f} {pytorch_ms/int8_ms:.2f}x")

Model                Size (MB)       Latency (ms)    Speedup
------------------------------------------------------------
PyTorch (FP32)       —               21.87           1.00x
ONNX (FP32)          42.83           10.14           2.16x
ONNX (INT8)          10.76           13.22           1.65x


| Model | File size (MB) | Avg latency (ms) | Speedup vs PyTorch |
|---|---|---|---|
| PyTorch (FP32) | - | 21.87 | 1.00× |
| ONNX (FP32) | 42.83 | 10.14 | 2.16x |
| ONNX (INT8) | 10.76 | 13.22 | 1.65x |  

The results were partially unexpected: ONNX FP32 was faster than INT8, which is unusual. Most of the speedup came from switching to ONNX Runtime itself rather than from quantization — ONNX Runtime's graph optimizations outperformed the INT8 weight compression benefit on CPU. The INT8 model, however, offers a 4x size reduction (42.83 MB → 10.76 MB), making it the better choice when storage or memory is the bottleneck rather than speed.